### Construção do dataset final

Espelha `src/pipeline/build_dataset.py`.

Fluxo:
1. Lê features intermediárias (`data/intermediate/sub-*.parquet`)
2. Faz merge com metadados (`participants.csv`)
3. Aplica filtro de amostra saudável
4. Gera dummies de metadados
5. Salva `data/processed/dataset_features.parquet`

Pré-requisito: ter rodado `01_build_networks.ipynb` (ou `extract_features.py`).


### Imports


In [ ]:
from pathlib import Path

import pandas as pd


: 

### Paths


In [ ]:
INTERMEDIATE_DIR = Path("../../data/intermediate")
METADATA_FILE = Path("../../data/metadata/participants.csv")
OUTPUT_DIR = Path("../../data/processed")
OUTPUT_FILE = OUTPUT_DIR / "dataset_features.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("INTERMEDIATE_DIR:", INTERMEDIATE_DIR.resolve())
print("METADATA_FILE:", METADATA_FILE.resolve())
print("OUTPUT_FILE:", OUTPUT_FILE.resolve())


### Carregar features intermediárias


In [ ]:
print("Lendo features intermediárias...")
files = sorted(INTERMEDIATE_DIR.glob("sub-*.parquet"))
print(f"Encontrados {len(files)} arquivos")

dfs = []
for file in files:
    subject_id, condition = file.stem.split("_", 1)
    df = pd.read_parquet(file)
    df["subject_id"] = subject_id
    df["condition"] = condition
    dfs.append(df)

df_features = pd.concat(dfs, ignore_index=True)
df_features.shape


### Merge com metadados


In [ ]:
print("Lendo metadados...")
meta = pd.read_csv(METADATA_FILE)
meta["subject_id"] = meta["ID"]

df_final = df_features.merge(
    meta,
    on="subject_id",
    how="left",
    validate="m:1",
)

In [ ]:
df_final

In [ ]:

df_final = df_final.rename(
    columns={
        "Gender_ 1=female_2=male": "gender",
        "Age": "age",
        "Handedness": "handedness",
        "Education": "education",
        "DRUG_0=negative_1=Positive": "drug",
        "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
        "SKID_Diagnoses": "skid_diagnoses",
        "SKID_Diagnoses 1": "skid_diagnoses_1",
        "SKID_Diagnoses 2": "skid_diagnoses_2",
        "Comments_SKID_assessment": "comments_skid_assessment",
        "Hamilton_Scale": "hamilton_scale",
        "BSL23_sumscore": "bsl23_sumscore",
        "BSL23_behavior": "bsl23_behavior",
        "AUDIT": "audit",
        "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
        "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
        "Relationship_Status": "relationship_status",
    }
)

print(f"Shape inicial: {df_final.shape}")
df_final[["subject_id", "condition", "age", "gender"]].head()


### Padronizar strings e filtrar amostra saudável


In [ ]:
df_final["relationship_status"] = df_final["relationship_status"].str.lower()
df_final["handedness"] = df_final["handedness"].str.lower()
df_final["education"] = df_final["education"].str.lower()

healthy_mask = (
    (df_final["drug"] == 0)
    & (df_final["smoking"] == 1)
    & (df_final["audit"] < 8)
    & (df_final["hamilton_scale"] <= 7)
    & (df_final["bsl23_behavior"] == 0)
    & (df_final["skid_diagnoses"].str.lower() == "none")
    & (df_final["skid_diagnoses_2"].isna())
    & (df_final["comments_skid_assessment"].isna())
)

df_healthy = df_final.loc[healthy_mask].copy()

print(f"Shape após filtro saudável: {df_healthy.shape}")
print(f"N indivíduos antes do filtro: {df_final['subject_id'].nunique()}")
print(f"N indivíduos após filtro saudável: {df_healthy['subject_id'].nunique()}")


### Dummies de metadados


In [ ]:
META_COLS = [
    "age",
    "gender",
    "handedness",
    "education",
    "relationship_status",
]

df_meta = df_healthy[META_COLS].copy()
print(f"Shape após seleção de metadados: {df_meta.shape}")

df_meta = pd.get_dummies(df_meta, columns=META_COLS, drop_first=True)
df_meta.head()


### Concatenar, corrigir E0 → EO e salvar


In [ ]:
df_analysis = pd.concat([df_healthy, df_meta], axis=1)
df_analysis["condition"] = df_analysis["condition"].replace({"E0": "EO"})

print(f"Shape após concatenação: {df_analysis.shape}")
print(df_analysis["condition"].value_counts())

df_analysis.to_parquet(OUTPUT_FILE, index=False)
print(f"Dataset final salvo em: {OUTPUT_FILE.resolve()}")
print(f"Shape final: {df_analysis.shape}")
df_analysis.head()


### Próximo passo

Com o dataset pronto, use os notebooks de EDA:

- `notebook/01_eda/01_eda_global.ipynb`
- `notebook/01_eda/02_eda_node_features.ipynb`
- `notebook/01_eda/03_analise_descriva_redes.ipynb`
